# Sentiment Analysis with Hugging Face Transformers

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch


classifier = pipeline('sentiment-analysis')


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

# Single Predictions

In [ ]:
print("\n--- Single Predictions ---")
print(classifier('We are very happy to show you the 🤗 Transformers library.'))
print(classifier('The pizza is not that great but the crust is awesome.'))


--- Single Predictions ---
[{'label': 'POSITIVE', 'score': 0.9997795224189758}]
[{'label': 'POSITIVE', 'score': 0.9998461008071899}]


# Batch Predictions

In [ ]:
print("\n--- Batch Predictions ---")
results = classifier([
    "We are very happy to show you the 🤗 Transformers library.",
    "We hope you don't hate it."
])

for result in results:
    print(f"label: {result['label']}, score: {round(result['score'], 4)}")


--- Batch Predictions ---
label: POSITIVE, score: 0.9998
label: NEGATIVE, score: 0.5309


# Custom Model Prediction

In [ ]:
print("\n--- Custom Model Prediction ---")

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

print(classifier("I am a good boy"))


--- Custom Model Prediction ---


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '4 stars', 'score': 0.42292678356170654}]


# Tokenization Example

In [ ]:
print("\n--- Tokenization Example ---")

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer("We are very happy to show you the 🤗 Transformers library.")
print(inputs)


--- Tokenization Example ---


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'input_ids': [101, 2057, 2024, 2200, 3407, 2000, 2265, 2017, 1996, 100, 19081, 3075, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


# Batch Tokenization (PyTorch tensors)

In [ ]:
# Batch tokenization (PyTorch tensors)
batch = tokenizer(
    [
        "We are very happy to show you the 🤗 Transformers library.",
        "We hope you don't hate it."
    ],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

for key, value in batch.items():
    print(f"{key}: {value}")

input_ids: tensor([[  101,  2057,  2024,  2200,  3407,  2000,  2265,  2017,  1996,   100,
         19081,  3075,  1012,   102],
        [  101,  2057,  3246,  2017,  2123,  1005,  1056,  5223,  2009,  1012,
           102,     0,     0,     0]])
token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


# Model Inference

In [ ]:
print("\n--- Model Inference ---")

with torch.no_grad():
    outputs = model(**batch)

print(outputs)


--- Model Inference ---
SequenceClassifierOutput(loss=None, logits=tensor([[-4.0833,  4.3364],
        [ 0.0818, -0.0418]]), hidden_states=None, attentions=None)


# Apply Softmax

In [ ]:
# Apply Softmax
probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(probabilities)

tensor([[2.2043e-04, 9.9978e-01],
        [5.3086e-01, 4.6914e-01]])


# Training Example

In [ ]:
print("\n--- Training Example ---")

labels = torch.tensor([1, 0])

outputs_with_loss = model(**batch, labels=labels)
print(outputs_with_loss)


--- Training Example ---
SequenceClassifierOutput(loss=tensor(0.3167, grad_fn=<NllLossBackward0>), logits=tensor([[-4.0833,  4.3364],
        [ 0.0818, -0.0418]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


# Saving Model

In [ ]:
print("\n--- Saving Model ---")

save_directory = "./saved_model"

tokenizer.save_pretrained(save_directory)
model.save_pretrained(save_directory)


--- Saving Model ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Loading Saved Model

In [ ]:
print("\n--- Loading Saved Model ---")

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(
    save_directory,
    attn_implementation='eager'
)


--- Loading Saved Model ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Hidden States & Attention

In [ ]:
print("\n--- Hidden States & Attention ---")

outputs = model(
    **batch,
    output_hidden_states=True,
    output_attentions=True,
    attn_implementation='eager'
)

all_hidden_states = outputs.hidden_states
all_attentions = outputs.attentions

print("Hidden states:", len(all_hidden_states))
print("Attention layers:", len(all_attentions))


--- Hidden States & Attention ---
Hidden states: 7
Attention layers: 6


## Inspecting Hidden States

In [ ]:
print("Type of all_hidden_states:", type(all_hidden_states))
print("Number of hidden layers:", len(all_hidden_states))

# Inspect the first hidden state
first_hidden_state = all_hidden_states[0]
print("Shape of the first hidden state:", first_hidden_state.shape)
print("First 5 elements of the first hidden state (first token, first embedding dimension):", first_hidden_state[0, 0, :5])

print("\nShape of each hidden state layer:")
for i, hidden_state_layer in enumerate(all_hidden_states):
    print(f"Layer {i}: {hidden_state_layer.shape}")

Type of all_hidden_states: <class 'tuple'>
Number of hidden layers: 7
Shape of the first hidden state: torch.Size([2, 14, 768])
First 5 elements of the first hidden state (first token, first embedding dimension): tensor([ 0.3549, -0.1386, -0.2253, -0.0478, -0.1200], grad_fn=<SliceBackward0>)

Shape of each hidden state layer:
Layer 0: torch.Size([2, 14, 768])
Layer 1: torch.Size([2, 14, 768])
Layer 2: torch.Size([2, 14, 768])
Layer 3: torch.Size([2, 14, 768])
Layer 4: torch.Size([2, 14, 768])
Layer 5: torch.Size([2, 14, 768])
Layer 6: torch.Size([2, 14, 768])


# Final Predictions

In [ ]:
print("\n=== FINAL PREDICTIONS ===")

classifier = pipeline('sentiment-analysis')

sample_texts = [
    "I love this product, it's amazing!",
    "This is the worst experience ever.",
    "It's okay, not bad but not great."
]

predictions = classifier(sample_texts)

for text, pred in zip(sample_texts, predictions):
    print(f"\nInput: {text}")
    print(f"Prediction: {pred['label']}")
    print(f"Confidence: {round(pred['score'], 4)}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.



=== FINAL PREDICTIONS ===


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Input: I love this product, it's amazing!
Prediction: POSITIVE
Confidence: 0.9999

Input: This is the worst experience ever.
Prediction: NEGATIVE
Confidence: 0.9998

Input: It's okay, not bad but not great.
Prediction: NEGATIVE
Confidence: 0.9978


# User Input Prediction

In [ ]:
print("\n--- User Input Prediction ---")

user_input = input("Enter a sentence: ")
result = classifier(user_input)

print("\nPrediction:", result[0]['label'])
print("Confidence:", round(result[0]['score'], 4))


--- User Input Prediction ---
Enter a sentence: i cant do this

Prediction: NEGATIVE
Confidence: 0.9913
